# import required sql functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *    

# Load Data From Datalake


### Load Customers data

In [0]:
df_customers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true")\
    .format("parquet") \
    .load("abfss://bronze@showroomdatalake.dfs.core.windows.net/mssql/customers/")

In [0]:
display(df_customers)

customer_id,customer_name,mobile,email,age,gender,address,city,created_date
1,rohan sharma,9810011221,Rohan.Sharma@Mail.in,34,Male,"101, Link Road, Phase 1",Mumbai,2025-01-01
2,Ananya Patel,9820011222,ananya.PATEL@yahoo.co.in,28,Female,"102, Link Road, Phase 2",Bangalore,2025-01-02
3,AMIT MISHRA,9830011223,AMIT.mishra@GMAIL.com,45,Male,"103, Link Road, Phase 3",Delhi,2025-01-03
4,Priya Reddy,9840011224,PriyaReddy@outlook.in,31,Female,"104, Link Road, Phase 4",Hyderabad,2025-01-04
5,vikram kumar,9810011225,vikram.KUMAR@Mail.in,52,Male,"105, Link Road, Phase 5",Mumbai,2025-01-05
6,Sneha Singh,9820011226,Sneha.Singh@icloud.com,24,Female,"106, Link Road, Phase 1",Bangalore,2025-01-06
7,RAHUL JOSHI,9830011227,rahul.JOSHI@rediffmail.com,39,Male,"107, Link Road, Phase 2",Delhi,2025-01-07
8,deepika nair,9840011228,Deepika.N@ProtonMail.com,41,Female,"108, Link Road, Phase 3",Hyderabad,2025-01-08
9,Arjun Gupta,9810011229,ARJUN.gupta@Mail.in,29,Male,"109, Link Road, Phase 4",Mumbai,2025-01-09
10,KAVITA MEHTA,9820011230,kavita.Mehta@Hotmail.com,36,Female,"110, Link Road, Phase 5",Bangalore,2025-01-10


### Transform Customer Table

In [0]:
df_customers.na.drop()

DataFrame[customer_id: int, customer_name: string, mobile: string, email: string, age: int, gender: string, address: string, city: string, created_date: date]

In [0]:
### drop duplicates
df_customers=df_customers.dropDuplicates()

In [0]:
display(df_customers)

customer_id,customer_name,mobile,email,age,gender,address,city,created_date
46,SNEHA SHARMA,9820011266,sneha.sharma@Outlook.in,25,Female,"146, Link Road, Phase 1",Bangalore,2025-02-15
76,divya patel,9840011296,divya.patel76@mail.in,25,Female,"176, Link Road, Phase 1",Hyderabad,2025-03-17
31,SANJAY SHARMA,9830011251,Sanjay.Sharma@gmail.com,49,Male,"131, Link Road, Phase 1",Delhi,2025-01-31
38,meera nair,9820011258,Meera.Nair@yahoo.co.in,47,Female,"138, Link Road, Phase 3",Bangalore,2025-02-07
66,sneha joshi,9820011286,sneha.joshi66@mail.in,27,Female,"166, Link Road, Phase 1",Bangalore,2025-03-07
80,KIRAN SINGH,9840011300,kiran.singh80@mail.in,23,Female,"180, Link Road, Phase 5",Hyderabad,2025-03-21
91,SANJAY PATEL,9830011311,sanjay.patel91@mail.in,46,Male,"191, Link Road, Phase 1",Delhi,2025-04-01
95,rajesh singh,9830011315,rajesh.singh95@mail.in,40,Male,"195, Link Road, Phase 5",Delhi,2025-04-05
28,DEEPIKA VERMA,9840011248,deepika.VERMA@gmail.com,40,Female,"128, Link Road, Phase 3",Hyderabad,2025-01-28
52,neha joshi,9840011272,neha.joshi52@mail.in,32,Female,"152, Link Road, Phase 2",Hyderabad,2025-02-21


In [0]:
###set all character at lower in email feild
df_customers=df_customers.withColumn('email',lower(col('email')))

In [0]:
## keep first character of name and surname capital and sort based on customer_id
df_customers=df_customers.withColumn('customer_name',initcap(col('customer_name')))\
.orderBy(col('customer_id'))


In [0]:
## add +91 before mobile number 
df_customers = df_customers.withColumn(
    "mobile",
    regexp_replace("mobile", "\\+91-\\+91", "+91-")
)


In [0]:
### add estimate date of birth based on age 
df_customers=df_customers.withColumn('Date_of_Birth',date_sub(current_date(),(col('age')*365)))

In [0]:
df_customers.display()

customer_id,customer_name,mobile,email,age,gender,address,city,created_date,Date_of_Birth
1,Rohan Sharma,+91-9810011221,rohan.sharma@mail.in,34,Male,"101, Link Road, Phase 1",Mumbai,2025-01-01,1992-07-11
2,Ananya Patel,+91-9820011222,ananya.patel@yahoo.co.in,28,Female,"102, Link Road, Phase 2",Bangalore,2025-01-02,1998-07-10
3,Amit Mishra,+91-9830011223,amit.mishra@gmail.com,45,Male,"103, Link Road, Phase 3",Delhi,2025-01-03,1981-07-14
4,Priya Reddy,+91-9840011224,priyareddy@outlook.in,31,Female,"104, Link Road, Phase 4",Hyderabad,2025-01-04,1995-07-11
5,Vikram Kumar,+91-9810011225,vikram.kumar@mail.in,52,Male,"105, Link Road, Phase 5",Mumbai,2025-01-05,1974-07-16
6,Sneha Singh,+91-9820011226,sneha.singh@icloud.com,24,Female,"106, Link Road, Phase 1",Bangalore,2025-01-06,2002-07-09
7,Rahul Joshi,+91-9830011227,rahul.joshi@rediffmail.com,39,Male,"107, Link Road, Phase 2",Delhi,2025-01-07,1987-07-13
8,Deepika Nair,+91-9840011228,deepika.n@protonmail.com,41,Female,"108, Link Road, Phase 3",Hyderabad,2025-01-08,1985-07-13
9,Arjun Gupta,+91-9810011229,arjun.gupta@mail.in,29,Male,"109, Link Road, Phase 4",Mumbai,2025-01-09,1997-07-10
10,Kavita Mehta,+91-9820011230,kavita.mehta@hotmail.com,36,Female,"110, Link Road, Phase 5",Bangalore,2025-01-10,1990-07-12


In [0]:
# Load data to Silver
df_customers.write \
    .mode("append") \
    .format("delta") \
    .option("path", "abfss://silver@showroomdatalake.dfs.core.windows.net/MSSQL/customers") \
    .save()

Read Sales data

In [0]:
df_sales=spark.read.format('parquet').option('inferSchema','true')\
    .option('header','true')\
        .load('abfss://bronze@showroomdatalake.dfs.core.windows.net/mssql/sales')

In [0]:
df_sales.display()

sale_id,invoice_number,customer_id,vin_number,car_model,base_cost,ex_showroom_price,payment_method,finance_provider,purchased_date,showroom_name,sales_executive
1,INV-2026-0001,34,1hgcr2f8xga000001,suv y,1500000.00,1800000.00,Bank Loan,HDFC Bank,2026-03-12,Branch Alpha,Rajesh Kumar
2,INV-2026-0002,12,1HGCR2F8XGA000002,Sedan X,1000000.00,1200000.00,Cash,null,2026-01-15,Branch Beta,Priya Das
3,INV-2026-0003,8,1hgcr2f8xga000003,Hatchback Z,500000.00,650000.00,Lease,Tata Capital,2026-05-20,Branch Gamma,Amit Patel
4,INV-2026-0004,47,1HGCR2F8XGA000004,suv y,1500000.00,1800000.00,Bank Loan,ICICI Bank,2026-02-25,Branch Alpha,Vikram Singh
5,INV-2026-0005,2,1hgcr2f8xga000005,SEDAN x,1000000.00,1200000.00,Cash,null,2026-06-02,Branch Beta,Rajesh Kumar
6,INV-2026-0006,25,1HGCR2F8xga000006,hatchback z,500000.00,650000.00,Bank Loan,SBI,2026-04-10,Branch Gamma,Priya Das
7,INV-2026-0007,19,1hgcr2f8XGA000007,SUV Y,1500000.00,1800000.00,Lease,Axis Bank,2026-01-18,Branch Alpha,Amit Patel
8,INV-2026-0008,41,1HGCR2F8XGA000008,sedan X,1000000.00,1200000.00,Cash,null,2026-03-28,Branch Beta,Vikram Singh
9,INV-2026-0009,5,1hgcr2f8xga000009,Hatchback Z,500000.00,650000.00,Bank Loan,HDFC Bank,2026-02-04,Branch Gamma,Rajesh Kumar
10,INV-2026-0010,30,1HGCR2F8XGA000010,suv Y,1500000.00,1800000.00,Cash,null,2026-06-12,Branch Alpha,Priya Das


### Transform Sales data

In [0]:
### Capital vin_number and car_model
df_sales=df_sales.withColumn('vin_number',upper(col('vin_number')))\
.withColumn('car_model',upper(col('car_model')))

In [0]:
### add default value for finance_provider
df_sales=df_sales.withColumn("finance_provider",when(col("finance_provider").isNull(), "Not Needed")\
    .otherwise(col("finance_provider")))

In [0]:
### calculate total_tax
df_sales=df_sales.withColumn('total_tax',(col('ex_showroom_price')-col('base_cost')))

In [0]:
#### calculate other charges such as RTO and other charges
df_sales=df_sales.withColumn('other_charges',(col('ex_showroom_price')*0.0205)+(col('base_cost')*0.025))

In [0]:
### Total On-road price
df_sales = df_sales.withColumn('on-road_price', (col('base_cost') + col('total_tax') + col('other_charges')))


In [0]:
df_sales.display()

sale_id,invoice_number,customer_id,vin_number,car_model,base_cost,ex_showroom_price,payment_method,finance_provider,purchased_date,showroom_name,sales_executive,total_tax,other_charges,on-road_price
4,INV-2026-0004,47,1HGCR2F8XGA000004,SUV Y,1500000.00,1800000.00,Bank Loan,ICICI Bank,2026-02-25,Branch Alpha,Vikram Singh,300000.00,74400.0,1874400.0
26,INV-2026-0026,45,1HGCR2F8XGA000026,SEDAN X,1000000.00,1200000.00,Cash,Not Needed,2026-02-08,Branch Beta,Priya Das,200000.00,49600.0,1249600.0
44,INV-2026-0044,26,1HGCR2F8XGA000044,SEDAN X,1000000.00,1200000.00,Cash,Not Needed,2026-06-27,Branch Beta,Vikram Singh,200000.00,49600.0,1249600.0
46,INV-2026-0046,35,1HGCR2F8XGA000046,SUV Y,1500000.00,1800000.00,Cash,Not Needed,2026-01-27,Branch Alpha,Priya Das,300000.00,74400.0,1874400.0
59,INV-2026-0059,5,1HGCR2F8XGA000059,SEDAN X,1000000.00,1200000.00,Bank Loan,SBI,2026-03-19,Branch Beta,Amit Patel,200000.00,49600.0,1249600.0
64,INV-2026-0064,19,1HGCR2F8XGA000064,SUV Y,1500000.00,1800000.00,Lease,Tata Capital,2025-12-22,Branch Alpha,Vikram Singh,300000.00,74400.0,1874400.0
76,INV-2026-0076,79,1HGCR2F8XGA000076,SUV Y,1500000.00,1800000.00,Lease,Kotak Mahindra,2026-01-10,Branch Alpha,Vikram Singh,300000.00,74400.0,1874400.0
81,INV-2026-0081,66,1HGCR2F8XGA000081,HATCHBACK Z,500000.00,650000.00,Bank Loan,HDFC Bank,2025-11-02,Branch Gamma,Rajesh Kumar,150000.00,25825.0,675825.0
85,INV-2026-0085,71,1HGCR2F8XGA000085,SUV Y,1500000.00,1800000.00,Cash,Not Needed,2025-07-25,Branch Alpha,Rajesh Kumar,300000.00,74400.0,1874400.0
87,INV-2026-0087,92,1HGCR2F8XGA000087,HATCHBACK Z,500000.00,650000.00,Cash,Not Needed,2025-12-27,Branch Gamma,Amit Patel,150000.00,25825.0,675825.0


In [0]:
df_sales=df_sales.dropDuplicates()

In [0]:
df_sales.describe().display()

summary,sale_id,invoice_number,customer_id,vin_number,car_model,base_cost,ex_showroom_price,payment_method,finance_provider,showroom_name,sales_executive,total_tax,other_charges,on-road_price
count,100,100,100,100,100,100,100,100,100,100,100,100,100,100
mean,50.5,null,37.8,null,null,1005000.000000,1222500.000000,null,null,null,null,217500.000000,50186.25,1272686.25
stddev,29.011491975882016,null,27.588242262078076,null,null,411298.7559751022,473242.3621500228,null,null,null,null,62915.28696058958,19982.38156268333,493223.16170226975
min,1,INV-2026-0001,1,1HGCR2F8XGA000002,HATCHBACK Z,500000.00,650000.00,Bank Loan,Axis Bank,Branch Alpha,Amit Patel,150000.00,25825.0,675825.0
max,100,INV-2026-0100,99,1HGCR2F8XGA000100,SUV Y,1500000.00,1800000.00,Lease,Tata Capital,Branch Gamma,Vikram Singh,300000.00,74400.0,1874400.0


### Load sales data to silver

In [0]:
df_sales.write.format('delta').mode('append')\
    .option('path','abfss://silver@showroomdatalake.dfs.core.windows.net/MSSQL/sales').save()

### READ SERVICE_LOG DATASET

In [0]:
df_sv_log=spark.read.format('parquet').option('inferSchema','true')\
    .option('header','true')\
        .load('abfss://bronze@showroomdatalake.dfs.core.windows.net/mssql/service_logs')

In [0]:
display(df_sv_log)

service_id,customer_id,vin_number,car_model,service_type,parts_cost,labor_cost,service_date,showroom_name,customer_rating
1,45,1hgcr2f8xga000051,Hatchback z,Routine Maintenance,1200.00,450.00,2025-10-14,Branch Gamma,5
2,12,1HGCR2F8XGA000052,suv y,Warranty Claim,0.00,1200.00,2026-02-18,Branch Alpha,4
3,89,1hgcr2f8xga000053,SEDAN x,Accidental Repair,15400.00,4500.00,2026-03-02,Branch Beta,3
4,3,1HGCR2F8XGA000054,Hatchback Z,Routine Maintenance,2100.00,600.00,2026-05-12,Branch Gamma,5
5,61,1hgcr2f8xga000055,suv y,Routine Maintenance,4300.00,900.00,2025-11-20,Branch Alpha,4
6,24,1HGCR2F8xga000056,sedan X,Accidental Repair,8900.00,3200.00,2026-01-15,Branch Beta,2
7,10,1hgcr2f8XGA000057,hatchback z,Warranty Claim,0.00,850.00,2026-04-10,Branch Gamma,5
8,77,1HGCR2F8XGA000008,SUV Y,Routine Maintenance,5100.00,1100.00,2025-12-05,Branch Alpha,4
9,5,1hgcr2f8xga000059,sedan x,Routine Maintenance,1800.00,500.00,2026-05-19,Branch Beta,5
10,92,1HGCR2F8XGA000060,Hatchback Z,Accidental Repair,22000.00,6500.00,2025-09-11,Branch Gamma,3


In [0]:
### Capital column vin_number and car_model
df_sv_log=df_sv_log.withColumn('vin_number', upper(col('vin_number'))).withColumn('car_model', upper(col('car_model')))

In [0]:
### Remove leading space from vin_number and car_model
df_sv_log = df_sv_log.withColumn("vin_number", ltrim(col("vin_number")))\
    .withColumn('car_model', ltrim(col('car_model')))

In [0]:
### add total cost column
df_sv_log=df_sv_log.withColumn('total_cost', (col('parts_cost'))+(col('labor_cost')))

In [0]:
df_sv_log.display()

service_id,customer_id,vin_number,car_model,service_type,parts_cost,labor_cost,service_date,showroom_name,customer_rating,total_cost
12,84,1HGCR2F8XGA000062,SEDAN X,Warranty Claim,0.00,1400.00,2026-03-22,Branch Beta,5,1400.00
13,31,1HGCR2F8XGA000063,HATCHBACK Z,Routine Maintenance,1450.00,450.00,2025-10-10,Branch Gamma,4,1900.00
16,4,1HGCR2F8XGA000066,HATCHBACK Z,Routine Maintenance,1100.00,400.00,2025-11-03,Branch Gamma,2,1500.00
34,40,1HGCR2F8XGA000084,HATCHBACK Z,Accidental Repair,9100.00,3100.00,2026-05-27,Branch Gamma,3,12200.00
60,91,1HGCR2F8XGA000060,HATCHBACK Z,Accidental Repair,19800.00,5800.00,2026-04-27,Branch Gamma,2,25600.00
78,94,1HGCR2F8XGA000078,HATCHBACK Z,Warranty Claim,0.00,800.00,2025-08-02,Branch Gamma,4,800.00
82,38,1HGCR2F8XGA000082,SUV Y,Routine Maintenance,5000.00,1200.00,2025-09-14,Branch Alpha,5,6200.00
83,6,1HGCR2F8XGA000083,SEDAN X,Warranty Claim,0.00,1100.00,2025-12-16,Branch Beta,4,1100.00
95,35,1HGCR2F8XGA000095,SEDAN X,Accidental Repair,13900.00,4200.00,2025-10-11,Branch Beta,3,18100.00
38,27,1HGCR2F8XGA000088,SUV Y,Routine Maintenance,5800.00,1400.00,2026-05-08,Branch Alpha,5,7200.00


In [0]:
df_sv_log=df_sv_log.dropDuplicates()

### Load Data Into Silver Container


In [0]:
df_sv_log.write.format('delta').mode('append')\
    .option('path','abfss://silver@showroomdatalake.dfs.core.windows.net/MSSQL/service_logs').save()

# Load data from bronze/postgress


In [0]:
df_mleads=spark.read.format('parquet').option('inferSchema','true')\
    .option('header','true')\
        .load('abfss://bronze@showroomdatalake.dfs.core.windows.net/postgres/marketing_leads')


In [0]:
df_mleads.display()

lead_id,first_name,last_name,email,phone,lead_source,campaign_name,lead_status,created_date
1,Rohan,Sharma,rohan.sharma.marketing@gmail.com,9810022331,Google_Ads,SUV Summer Launch,New,2026-06-28T11:53:34.083Z
2,SNEHA,patel,sneha.patel99@outlook.com,9820022332,facebook ads,Sedan Discount Wave,Contacted,2026-06-28T11:53:34.083Z
3,amit,MISHRA,AMIT.MISHRA@HOTMAIL.COM,9830022333,GOOGLE ADS,SUV Summer Launch,Junk,2026-06-28T11:53:34.083Z
4,Priya,Reddy,priya.reddy.lead@yahoo.com,9840022334,Instagram,Hatchback College Promo,New,2026-06-28T11:53:34.083Z
5,VIKRAM,kumar,vikram.k@domain.in,9810022335,Facebook_Ads,Sedan Discount Wave,New,2026-06-28T11:53:34.083Z
6,Ananya,Singh,ananya.singh.marketing@gmail.com,9820022336,google_ads,SUV Summer Launch,Contacted,2026-06-28T11:53:34.083Z
7,rahul,Joshi,rahul.j@outlook.in,9830022337,LinkedIn,B2B Corporate Fleet,New,2026-06-28T11:53:34.083Z
8,Deepika,NAIR,DEEPIKA.NAIR@PROTON.ME,9840022338,Facebook Ads,Hatchback College Promo,Junk,2026-06-28T11:53:34.083Z
9,ARJUN,GUPTA,arjun.g@gmail.com,9810022339,Google Ads,SUV Summer Launch,Contacted,2026-06-28T11:53:34.083Z
10,kavita,Mehta,kavita.mehta@yahoo.co.in,9820022340,instagram,Hatchback College Promo,New,2026-06-28T11:53:34.083Z


In [0]:
df_mleads=df_mleads.dropDuplicates()

In [0]:
df_mleads=df_mleads.withColumn('first_name',initcap('first_name')).withColumn('last_name',initcap('last_name'))\
    .withColumn('email',lower('email'))

In [0]:
df_mleads = df_mleads.withColumn("phone",concat(lit("+91-"), col("phone")))

In [0]:
df_mleads=df_mleads.withColumn('customer_id',col('lead_id'))

In [0]:
df_mleads.display()

lead_id,first_name,last_name,email,phone,lead_source,campaign_name,lead_status,created_date,customer_id
2,Sneha,Patel,sneha.patel99@outlook.com,+91-9820022332,facebook ads,Sedan Discount Wave,Contacted,2026-06-28T11:53:34.083Z,2
14,Pooja,Rao,pooja.rao90@gmail.com,+91-9820022344,Google Ads,SUV Summer Launch,New,2026-06-28T11:53:34.083Z,14
17,Gaurav,Patel,gaurav.p.marketing@gmail.com,+91-9810022347,google_ads,SUV Summer Launch,New,2026-06-28T11:53:34.083Z,17
43,Amit,Verma,amit.verma@yahoo.com,+91-9830022373,Google Ads,Sedan Discount Wave,Junk,2026-06-28T11:53:34.083Z,43
60,Kavita,Mehta,kavita.m@gmail.com,+91-9820022390,Instagram,Hatchback College Promo,Junk,2026-06-28T11:55:37.681Z,60
72,Ananya,Joshi,ananya.j@gmail.com,+91-9820022402,Instagram,Hatchback College Promo,New,2026-06-28T11:55:37.681Z,72
94,Priya,Rao,priya.rao@outlook.in,+91-9840022424,LinkedIn,B2B Corporate Fleet,New,2026-06-28T11:55:37.681Z,94
96,Sneha,Sharma,sneha.s@yahoo.co.in,+91-9820022426,Facebook Ads,Sedan Discount Wave,Contacted,2026-06-28T11:55:37.681Z,96
33,Aditya,Mishra,aditya.m@yahoo.com,+91-9810022363,LinkedIn,B2B Corporate Fleet,Contacted,2026-06-28T11:53:34.083Z,33
39,Vivek,Gupta,vivek.g@yahoo.com,+91-9830022369,LinkedIn,B2B Corporate Fleet,Contacted,2026-06-28T11:53:34.083Z,39


# APPLY JOINS TO FIND INSIGHTS 

In [0]:
df=df_mleads.join(df_sales, on='customer_id', how='inner')


### # load 


In [0]:
df_mleads.write.mode('overwrite').format('delta')\
    .option('path','abfss://silver@showroomdatalake.dfs.core.windows.net/postgress/marketing_leads')